# Run Multi-Agent Pipeline On Benchmark 200

This notebook is wired for the Kaggle dataset layout shown in the input panel:

- Code: set `PIPELINE_MA_DIR`, place this notebook inside the pipeline directory, or let it auto-detect a folder containing `run_benchmark_model_ablation.py` under Kaggle/Colab inputs.
- Data: `/kaggle/input/indexing-parquet/pac_test_topics_benchmark_200.parquet` or `/kaggle/input/datasets/djnhngocduc/indexing-parquet/pac_test_topics_benchmark_200.parquet`
- Qrels: `/kaggle/input/indexing-parquet/pac_test_qrels_benchmark_200.parquet` or `/kaggle/input/datasets/djnhngocduc/indexing-parquet/pac_test_qrels_benchmark_200.parquet`

The benchmark cells run only `PB2`: Query Understanding Agent + fixed KNN retrieval with a Groq cross-family model ablation. The `PB1` baseline is the RAG-based notebook result, so this notebook does not rerun PB1 for the benchmark table. The optional debug section below can run one topic step by step through Agent 1, retrieval, Agent 2, Agent 3, and output.

In [ ]:
import os
import sys
import json
import shutil
import subprocess
import time
from pathlib import Path

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ.setdefault('HF_HOME', '/kaggle/working/hf_cache')
os.environ.setdefault('TRANSFORMERS_CACHE', os.environ['HF_HOME'])

def first_existing(paths, label):
    checked = []
    for item in paths:
        if not item:
            continue
        p = Path(str(item)).expanduser()
        checked.append(str(p))
        if p.exists():
            return p.resolve()
    raise FileNotFoundError(f'Cannot find {label}. Checked:\n' + '\n'.join(checked))

def find_pipeline_dir():
    marker = 'run_benchmark_model_ablation.py'
    direct = [
        os.getenv('PIPELINE_MA_DIR'),
        Path.cwd(),
        Path.cwd() / 'pipeline-ma',
        Path.cwd().parent / 'pipeline-ma',
        '/kaggle/working/pipeline-ma',
        '/kaggle/working/Project/pipeline-ma',
        '/content/drive/MyDrive/Project/pipeline-ma',
    ]
    for item in direct:
        if not item:
            continue
        p = Path(str(item)).expanduser()
        if (p / marker).exists():
            return p.resolve()
    search_roots = [Path('/kaggle/input'), Path('/kaggle/working'), Path('/content/drive/MyDrive')]
    for root in search_roots:
        if not root.exists():
            continue
        for match in root.rglob(marker):
            return match.parent.resolve()
    raise FileNotFoundError('Cannot find multi-agent pipeline directory. Set PIPELINE_MA_DIR.')

def patch_runtime_pipeline_source(path):
    module_path = path / 'patent_multiagent_langgraph.py'
    if not module_path.exists():
        return
    text = module_path.read_text(encoding='utf-8')
    original = text
    text = text.replace(
        '"json mode",\n                "invalid parameter",',
        '"json mode",\n                "failed to validate json",\n                "json_validate_failed",\n                "failed_generation",\n                "invalid parameter",',
    )
    if text != original:
        module_path.write_text(text, encoding='utf-8')
        print('Patched Groq JSON validation retry in', module_path)
    req_path = path / 'requirements-multiagent.txt'
    if req_path.exists():
        req_text = req_path.read_text(encoding='utf-8')
        patched_req = req_text.replace('transformers>=4.51.0', 'transformers==4.45.2')
        patched_req = patched_req.replace('transformers>=4.45.0', 'transformers==4.45.2')
        if patched_req != req_text:
            req_path.write_text(patched_req, encoding='utf-8')
            print('Pinned transformers==4.45.2 in', req_path)


PIPELINE_DIR = find_pipeline_dir()
if str(PIPELINE_DIR).startswith('/kaggle/input/'):
    runtime_dir = Path('/kaggle/working/pipeline-ma-runtime')
    if not runtime_dir.exists():
        shutil.copytree(PIPELINE_DIR, runtime_dir)
    PIPELINE_DIR = runtime_dir.resolve()
    os.environ['PIPELINE_MA_DIR'] = str(PIPELINE_DIR)

patch_runtime_pipeline_source(PIPELINE_DIR)

if not (PIPELINE_DIR / 'run_benchmark_model_ablation.py').exists():
    raise FileNotFoundError(f'Wrong PIPELINE_DIR: {PIPELINE_DIR}')

sys.path.insert(0, str(PIPELINE_DIR))
sys.path.insert(0, str(PIPELINE_DIR.parent))

# Force imports to come from PIPELINE_DIR after sys.path changes.
for module_name in ['patent_multiagent_langgraph', 'run_benchmark_model_ablation', 'evaluate_ablation']:
    sys.modules.pop(module_name, None)

print('PIPELINE_DIR  =', PIPELINE_DIR)

TOPICS_PATH = first_existing([
    os.getenv('BENCHMARK_TOPICS_PATH'),
    os.getenv('MULTIAGENT_BENCHMARK_TOPICS_PATH'),
    '/kaggle/input/datasets/djnhngocduc/indexing-parquet/pac_test_topics_benchmark_200.parquet',
    '/kaggle/input/indexing-parquet/pac_test_topics_benchmark_200.parquet',
    '/kaggle/working/clefip2011_pac_topics/processed/pac_test_topics_benchmark_200.parquet',
], 'benchmark_200 topics parquet')

QRELS_PATH = first_existing([
    os.getenv('BENCHMARK_QRELS_PATH'),
    os.getenv('MULTIAGENT_BENCHMARK_QRELS_PATH'),
    '/kaggle/input/datasets/djnhngocduc/indexing-parquet/pac_test_qrels_benchmark_200.parquet',
    '/kaggle/input/indexing-parquet/pac_test_qrels_benchmark_200.parquet',
    '/kaggle/working/clefip2011_pac_topics/processed/pac_test_qrels_benchmark_200.parquet',
], 'benchmark_200 qrels parquet')

OUTPUT_DIR = Path(os.getenv('MA_OUTPUT_DIR', '/kaggle/working/multiagent_benchmark_200')).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

os.environ['BENCHMARK_TOPICS_PATH'] = str(TOPICS_PATH)
os.environ['BENCHMARK_QRELS_PATH'] = str(QRELS_PATH)

print('PIPELINE_DIR =', PIPELINE_DIR)
print('TOPICS_PATH  =', TOPICS_PATH)
print('QRELS_PATH   =', QRELS_PATH)
print('OUTPUT_DIR   =', OUTPUT_DIR)

In [ ]:
# Install MA dependencies for Agent 1. Retrieval embedding server is started later, before Stage 2.
INSTALL_DEPS = os.getenv('INSTALL_DEPS', 'true').lower() == 'true'
if INSTALL_DEPS:
    req = PIPELINE_DIR / 'requirements-multiagent.txt'
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req)])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', 'transformers==4.45.2'])
    print('Pinned transformers==4.45.2 for Jina v3 remote-code embeddings')
else:
    print('Skipped dependency install')

In [ ]:
# Load secrets from Kaggle Secrets when available.
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    for name in ['GROQ_API_KEY', 'ES_CLOUD_ID', 'ES_API_KEY', 'ES_USER', 'ES_PASSWORD', 'HF_TOKEN', 'HUGGINGFACEHUB_API_TOKEN']:
        if os.getenv(name):
            continue
        try:
            value = secrets.get_secret(name)
            if value:
                os.environ[name] = value
        except Exception:
            pass
except Exception:
    pass

if os.getenv('HF_TOKEN') and not os.getenv('HUGGINGFACEHUB_API_TOKEN'):
    os.environ['HUGGINGFACEHUB_API_TOKEN'] = os.environ['HF_TOKEN']

print('Has GROQ_API_KEY:', bool(os.getenv('GROQ_API_KEY')))
print('Has ES_CLOUD_ID:', bool(os.getenv('ES_CLOUD_ID')))
print('Has ES_API_KEY :', bool(os.getenv('ES_API_KEY')))
print('Has HF token   :', bool(os.getenv('HF_TOKEN') or os.getenv('HUGGINGFACEHUB_API_TOKEN')))

In [ ]:
# Core runtime settings. Keep retrieval fixed to the RAG-based KNN index.
os.environ['MULTIAGENT_RETRIEVAL_BACKEND'] = 'es_knn'
os.environ.setdefault('BM25_INDEX', 'clef_ip_patents_v1_mini')
os.environ.setdefault('KNN_INDEX', 'clef_ip_patents_v1_mini_jina')
os.environ.setdefault('ES_VECTOR_FIELD', 'content_vector')

# Groq free API cores for PB2 model ablation: one GPT-OSS, one Qwen, one Llama.
# Limits differ by model, so the run applies a conservative common 6K TPM cap.
# Groq free daily token caps still apply; use MA_LIMIT/resume to split full 200-topic runs if needed.
FAST_MODE = os.getenv('MA_FAST_MODE', 'false').lower() == 'true'
if FAST_MODE:
    # Fast smoke mode: validate Agent 1 -> retrieval with one 8K-min model first.
    os.environ['MULTIAGENT_BENCHMARK_MODELS'] = os.getenv('MULTIAGENT_FAST_MODE_MODEL', 'openai/gpt-oss-120b')
    os.environ.setdefault('MA_LIMIT', '20')

os.environ['MULTIAGENT_MODEL_PRESET'] = os.getenv('MULTIAGENT_MODEL_PRESET', 'groq-free-diverse-common')
os.environ['MULTIAGENT_BENCHMARK_MODELS'] = os.getenv(
    'MULTIAGENT_BENCHMARK_MODELS',
    'openai/gpt-oss-120b,qwen/qwen3-32b,llama-3.3-70b-versatile',
)
os.environ['MULTIAGENT_LLM_PROVIDER'] = 'groq'
os.environ['MULTIAGENT_LLM_MODEL'] = os.getenv('MULTIAGENT_LLM_MODEL', 'openai/gpt-oss-120b')
os.environ['MULTIAGENT_AGENT2_LLM_MODEL'] = os.getenv('MULTIAGENT_AGENT2_LLM_MODEL', 'openai/gpt-oss-120b')
os.environ['MULTIAGENT_AGENT3_LLM_MODEL'] = os.getenv('MULTIAGENT_AGENT3_LLM_MODEL', 'openai/gpt-oss-120b')
agent2_out_tokens = min(int(os.getenv('MULTIAGENT_AGENT2_MAX_OUTPUT_TOKENS', '1200') or '1200'), 1200)
os.environ['MULTIAGENT_AGENT2_MAX_OUTPUT_TOKENS'] = str(agent2_out_tokens)
agent3_out_tokens = min(int(os.getenv('MULTIAGENT_AGENT3_MAX_OUTPUT_TOKENS', '1000') or '1000'), 1000)
os.environ['MULTIAGENT_AGENT3_MAX_OUTPUT_TOKENS'] = str(agent3_out_tokens)
os.environ['MULTIAGENT_GROQ_TOKEN_PACING'] = os.getenv('MULTIAGENT_GROQ_TOKEN_PACING', 'true')
os.environ['MULTIAGENT_GROQ_GPT_OSS_REASONING_EFFORT'] = os.getenv('MULTIAGENT_GROQ_GPT_OSS_REASONING_EFFORT', 'low')
os.environ['MULTIAGENT_GROQ_QWEN_REASONING_EFFORT'] = os.getenv('MULTIAGENT_GROQ_QWEN_REASONING_EFFORT', 'none')
os.environ['MULTIAGENT_GROQ_INCLUDE_REASONING'] = os.getenv('MULTIAGENT_GROQ_INCLUDE_REASONING', 'false')
os.environ['MULTIAGENT_LLM_PLAIN_RETRY_DISABLED'] = os.getenv('MULTIAGENT_LLM_PLAIN_RETRY_DISABLED', 'false')
os.environ['MULTIAGENT_GROQ_TPM_LIMIT'] = os.getenv('MULTIAGENT_GROQ_TPM_LIMIT', '6000')
groq_margin = max(int(os.getenv('MULTIAGENT_GROQ_TPM_MARGIN_TOKENS', '0') or '0'), 900)
os.environ['MULTIAGENT_GROQ_TPM_MARGIN_TOKENS'] = str(groq_margin)
os.environ['MULTIAGENT_GROQ_PROMPT_TOKEN_SAFETY_FACTOR'] = os.getenv('MULTIAGENT_GROQ_PROMPT_TOKEN_SAFETY_FACTOR', '1.25')
os.environ['MULTIAGENT_LLM_TEMPERATURE'] = '0'
os.environ['MULTIAGENT_LLM_STRICT'] = 'true'
os.environ['MULTIAGENT_LLM_JSON_SCHEMA_STRICT'] = 'false'
os.environ['MULTIAGENT_LLM_JSON_OBJECT_MODE'] = 'false'
os.environ['MULTIAGENT_LLM_JSON_REPAIR'] = 'false'
os.environ['MULTIAGENT_BENCHMARK_SLEEP_SECONDS'] = os.getenv('MULTIAGENT_BENCHMARK_SLEEP_SECONDS', '15' if FAST_MODE else '30')
os.environ['MULTIAGENT_AGENT1_INPUT_WORDS'] = os.getenv('MULTIAGENT_AGENT1_INPUT_WORDS', '220')
os.environ['MULTIAGENT_AGENT1_TITLE_WORDS'] = os.getenv('MULTIAGENT_AGENT1_TITLE_WORDS', '40')
os.environ['MULTIAGENT_AGENT1_ABSTRACT_WORDS'] = os.getenv('MULTIAGENT_AGENT1_ABSTRACT_WORDS', '90')
os.environ['MULTIAGENT_AGENT1_CLAIMS_WORDS'] = os.getenv('MULTIAGENT_AGENT1_CLAIMS_WORDS', '90')
os.environ['MULTIAGENT_AGENT1_RETRIEVAL_TEXT_WORDS'] = os.getenv('MULTIAGENT_AGENT1_RETRIEVAL_TEXT_WORDS', '50')
agent1_out_tokens = min(int(os.getenv('MULTIAGENT_AGENT1_MAX_OUTPUT_TOKENS', '900') or '900'), 900)
agent1_out_tokens = max(agent1_out_tokens, 700)
os.environ['MULTIAGENT_AGENT1_MAX_OUTPUT_TOKENS'] = str(agent1_out_tokens)
os.environ['MULTIAGENT_AGENT1_METADATA_MAX_KEYS'] = os.getenv('MULTIAGENT_AGENT1_METADATA_MAX_KEYS', '24')
os.environ['MULTIAGENT_AGENT1_METADATA_WORDS'] = os.getenv('MULTIAGENT_AGENT1_METADATA_WORDS', '220')
llm_out_tokens = min(max(int(os.getenv('MULTIAGENT_LLM_MAX_OUTPUT_TOKENS', '900') or '900'), agent1_out_tokens), 900)
os.environ['MULTIAGENT_LLM_MAX_OUTPUT_TOKENS'] = str(llm_out_tokens)
os.environ['MULTIAGENT_EVIDENCE_TOP_DOCS'] = os.getenv('MULTIAGENT_EVIDENCE_TOP_DOCS', '3')
os.environ['MULTIAGENT_AGENT2_TOP_DOCS'] = os.getenv('MULTIAGENT_AGENT2_TOP_DOCS', '3')
agent2_text_words = min(int(os.getenv('MULTIAGENT_AGENT2_CANDIDATE_TEXT_WORDS', '130') or '130'), 130)
os.environ['MULTIAGENT_AGENT2_CANDIDATE_TEXT_WORDS'] = str(agent2_text_words)
os.environ['MULTIAGENT_AGENT2_CLAIM_ELEMENTS'] = os.getenv('MULTIAGENT_AGENT2_CLAIM_ELEMENTS', '3')
agent2_ctx_chars = min(int(os.getenv('MULTIAGENT_AGENT2_RETRIEVAL_CONTEXT_CHARS', '500') or '500'), 500)
os.environ['MULTIAGENT_AGENT2_RETRIEVAL_CONTEXT_CHARS'] = str(agent2_ctx_chars)
os.environ['MULTIAGENT_AGENT3_TOP_DOCS'] = os.getenv('MULTIAGENT_AGENT3_TOP_DOCS', '3')
agent3_matches = min(int(os.getenv('MULTIAGENT_AGENT3_MATCHES_PER_DOC', '2') or '2'), 2)
os.environ['MULTIAGENT_AGENT3_MATCHES_PER_DOC'] = str(agent3_matches)
os.environ['MULTIAGENT_AGENT3_CANDIDATES_JSON_CHARS'] = os.getenv('MULTIAGENT_AGENT3_CANDIDATES_JSON_CHARS', '1200')
os.environ['MULTIAGENT_AGENT3_EVIDENCE_JSON_CHARS'] = os.getenv('MULTIAGENT_AGENT3_EVIDENCE_JSON_CHARS', '2600')
os.environ['MULTIAGENT_AGENT3_QUERY_JSON_CHARS'] = os.getenv('MULTIAGENT_AGENT3_QUERY_JSON_CHARS', '1200')

# Retrieval size aligned with RAG-based benchmark output top-300.
os.environ['MULTIAGENT_KNN_TOP_K'] = os.getenv('MULTIAGENT_KNN_TOP_K', '300')
os.environ['MULTIAGENT_KNN_NUM_CANDIDATES'] = os.getenv('MULTIAGENT_KNN_NUM_CANDIDATES', '1500')
os.environ['MULTIAGENT_KNN_MAX_FETCH_SIZE'] = os.getenv('MULTIAGENT_KNN_MAX_FETCH_SIZE', '1500')
os.environ['MULTIAGENT_KNN_CHUNK_FETCH_MULTIPLIER'] = os.getenv('MULTIAGENT_KNN_CHUNK_FETCH_MULTIPLIER', '12')
os.environ['VARIANT_FETCH_MULTIPLIER'] = os.getenv('VARIANT_FETCH_MULTIPLIER', '3')
os.environ['MULTIAGENT_KNN_VARIANT_FETCH_MULTIPLIER'] = os.getenv('MULTIAGENT_KNN_VARIANT_FETCH_MULTIPLIER', os.environ['VARIANT_FETCH_MULTIPLIER'])
os.environ['MULTIAGENT_KNN_SCORE_AGG'] = os.getenv('MULTIAGENT_KNN_SCORE_AGG', 'max')
os.environ['MULTIAGENT_KNN_RRF_K'] = os.getenv('MULTIAGENT_KNN_RRF_K', '60')
os.environ['MULTIAGENT_KNN_HF_MODEL'] = os.getenv('MULTIAGENT_KNN_HF_MODEL', 'jinaai/jina-embeddings-v3')
os.environ['MULTIAGENT_KNN_EMBED_TASK'] = os.getenv('MULTIAGENT_KNN_EMBED_TASK', 'retrieval.query')

RUN_VARIANTS = [x.strip().lower() for x in os.getenv('RUN_VARIANTS', 'pb2').split(',') if x.strip()]
bad = sorted(set(RUN_VARIANTS) - {'pb2'})
if bad:
    raise ValueError(f'This notebook is for PB2 only. Unsupported RUN_VARIANTS: {bad}')

K_VALUES = os.getenv('MA_K_VALUES', '5,10,20,50,100')
RANKING_SOURCE = os.getenv('MA_RANKING_SOURCE', 'candidates')
LIMIT = os.getenv('MA_LIMIT', '0')
SAVE_EVERY = os.getenv('MA_SAVE_EVERY', '1')
TEXT_COLUMNS = os.getenv('MA_TEXT_COLUMNS', 'title,abstract')

print('RUN_VARIANTS  =', RUN_VARIANTS)
print('FAST_MODE     =', FAST_MODE)
print('MODEL_PRESET  =', os.environ['MULTIAGENT_MODEL_PRESET'])
print('MODEL_OVERRIDE=', os.getenv('MULTIAGENT_BENCHMARK_MODELS', ''))
print('AGENT2_MODEL  =', os.environ['MULTIAGENT_AGENT2_LLM_MODEL'])
print('AGENT3_MODEL  =', os.environ['MULTIAGENT_AGENT3_LLM_MODEL'])
print('AGENT1_OUTTOK =', os.environ['MULTIAGENT_AGENT1_MAX_OUTPUT_TOKENS'])
print('LLM_OUTTOK    =', os.environ['MULTIAGENT_LLM_MAX_OUTPUT_TOKENS'])
print('GPTOSS_REASON =', os.environ['MULTIAGENT_GROQ_GPT_OSS_REASONING_EFFORT'])
print('AGENT2_OUTTOK =', os.environ['MULTIAGENT_AGENT2_MAX_OUTPUT_TOKENS'])
print('AGENT3_OUTTOK =', os.environ['MULTIAGENT_AGENT3_MAX_OUTPUT_TOKENS'])
print('AGENT2_CAPS   =', os.environ['MULTIAGENT_AGENT2_TOP_DOCS'], 'docs x', os.environ['MULTIAGENT_AGENT2_CANDIDATE_TEXT_WORDS'], 'words')
print('AGENT3_CAPS   =', os.environ['MULTIAGENT_AGENT3_TOP_DOCS'], 'docs x', os.environ['MULTIAGENT_AGENT3_MATCHES_PER_DOC'], 'matches')
print('SLEEP_SECONDS =', os.environ['MULTIAGENT_BENCHMARK_SLEEP_SECONDS'])
print('K_VALUES      =', K_VALUES)
print('RANKING_SRC   =', RANKING_SOURCE)
print('MA_LIMIT      =', LIMIT)
print('TEXT_COLUMNS  =', TEXT_COLUMNS)
print('KNN_INDEX     =', os.environ['KNN_INDEX'])
print('KNN_HF_MODEL  =', os.environ['MULTIAGENT_KNN_HF_MODEL'])
print('KNN_EMBED_API =', os.getenv('MULTIAGENT_KNN_EMBED_API_BASE', ''))

In [ ]:
# Verify transformer version before local Jina embedding load.
# If this prints a version other than 4.45.2 after pip install, restart the kernel and rerun from the top.
import transformers
print('TRANSFORMERS_VERSION =', transformers.__version__)
if transformers.__version__ != '4.45.2':
    raise RuntimeError('Expected transformers==4.45.2 for jinaai/jina-embeddings-v3. Restart kernel and rerun install/setup cells.')

In [ ]:
# Validate benchmark_200 files before spending GPU time.
import pandas as pd

topics = pd.read_parquet(TOPICS_PATH)
qrels = pd.read_parquet(QRELS_PATH)
topic_count = topics['topic_id'].astype(str).nunique()
qrel_topic_count = qrels['topic_id'].astype(str).nunique()
print('topics rows      =', len(topics))
print('unique topics    =', topic_count)
print('qrels rows       =', len(qrels))
print('qrels topics     =', qrel_topic_count)
if topic_count != 200 or qrel_topic_count != 200:
    raise RuntimeError('This is not the benchmark_200 topics/qrels pair.')
display(topics.head(3))

In [ ]:
# Runtime check. If ok=false only because LangGraph is missing, the benchmark runner still uses the linear path.
import inspect
import patent_multiagent_langgraph as ma
from patent_multiagent_langgraph import PipelineConfig, validate_runtime_environment

print('MODULE_FILE   =', ma.__file__)
print('PATCH_VERSION =', getattr(ma, 'RUNTIME_PATCH_VERSION', 'missing'))
print('AGENT1_SIG    =', inspect.signature(ma.run_query_understanding_stage))
if getattr(ma, 'RUNTIME_PATCH_VERSION', '') != '2026-05-26-canonical-vi-report-v12':
    raise RuntimeError('Stale runtime module loaded. Delete /kaggle/working/pipeline-ma-runtime, restart kernel, then rerun setup.')

check = validate_runtime_environment(PipelineConfig(pipeline_variant=RUN_VARIANTS[0]))
print(json.dumps(check, indent=2, ensure_ascii=False))
blocking = [issue for issue in check.get('issues', []) if 'LangGraph' not in issue and 'langgraph' not in issue]
if blocking:
    print('\nBlocking issues to fix before run:')
    for issue in blocking:
        print('-', issue)

## Single-Topic Step-By-Step Pipeline

Use these cells to run one topic through the main pipeline one step at a time: setup, Agent 1, retrieval, Agent 2, Agent 3, and output. Set `MULTIAGENT_DEBUG_TOPIC_INDEX` or `MULTIAGENT_DEBUG_MODEL` if needed.

In [ ]:
# Setup: select one topic and one Agent 1 model for a step-by-step run.
from IPython.display import Markdown, display

from evaluate_ablation import load_qrels
from run_benchmark_model_ablation import load_topics, parse_csv, resolve_models, selected_topics, jsonable
from patent_multiagent_langgraph import (
    PipelineConfig,
    RUNTIME_PATCH_VERSION,
    run_analysis_stage,
    run_evidence_stage,
    run_output_stage,
    run_query_understanding_stage,
    run_retrieval_stage,
)
print('RUNTIME_PATCH_VERSION =', RUNTIME_PATCH_VERSION)
if RUNTIME_PATCH_VERSION != '2026-05-26-canonical-vi-report-v12':
    raise RuntimeError('Stale patent_multiagent_langgraph.py loaded. Delete /kaggle/working/pipeline-ma-runtime, restart kernel, and rerun setup.')


def env_int(name, default):
    try:
        return int(os.getenv(name, str(default)))
    except Exception:
        return default


def env_bool(name, default=False):
    raw = os.getenv(name)
    if raw is None:
        return default
    return raw.lower() in {'1', 'true', 'yes', 'y', 'on'}


def slug(value):
    return ''.join(ch if ch.isalnum() or ch in {'-', '_'} else '_' for ch in str(value))[:120]


DEBUG_TOPIC_INDEX = env_int('MULTIAGENT_DEBUG_TOPIC_INDEX', 0)
DEBUG_MODELS = resolve_models(os.getenv('MULTIAGENT_BENCHMARK_MODELS', ''), os.environ['MULTIAGENT_MODEL_PRESET'])
DEBUG_MODEL = os.getenv('MULTIAGENT_DEBUG_MODEL', DEBUG_MODELS[0])

debug_rows = load_topics(TOPICS_PATH)
debug_qrels = load_qrels(QRELS_PATH)
debug_topics = selected_topics(
    debug_rows,
    debug_qrels,
    parse_csv(TEXT_COLUMNS),
    limit=DEBUG_TOPIC_INDEX + 1,
    allow_missing_qrels=True,
)
if DEBUG_TOPIC_INDEX >= len(debug_topics):
    raise IndexError(f'MULTIAGENT_DEBUG_TOPIC_INDEX={DEBUG_TOPIC_INDEX} but only {len(debug_topics)} topics are available')

debug_topic = debug_topics[DEBUG_TOPIC_INDEX]
debug_config = PipelineConfig(
    pipeline_variant=RUN_VARIANTS[0],
    llm_provider=os.environ['MULTIAGENT_LLM_PROVIDER'],
    llm_model=DEBUG_MODEL,
    llm_temperature=0.0,
    llm_max_tokens=env_int('MULTIAGENT_LLM_MAX_OUTPUT_TOKENS', 900),
    retrieval_backend=os.environ['MULTIAGENT_RETRIEVAL_BACKEND'],
    retrieval_strict=env_bool('MULTIAGENT_DEBUG_RETRIEVAL_STRICT', env_bool('MULTIAGENT_RETRIEVAL_STRICT', True)),
    candidate_screen_top_k=env_int('MULTIAGENT_CANDIDATE_TOP_K', 20),
    evidence_top_docs=env_int('MULTIAGENT_EVIDENCE_TOP_DOCS', 3),
    max_iterations=env_int('MULTIAGENT_DEBUG_MAX_ITERATIONS', 1),
    knn_top_k=env_int('MULTIAGENT_KNN_TOP_K', 300),
    knn_num_candidates=env_int('MULTIAGENT_KNN_NUM_CANDIDATES', 1500),
    knn_chunk_fetch_multiplier=env_int('MULTIAGENT_KNN_CHUNK_FETCH_MULTIPLIER', 12),
    knn_variant_fetch_multiplier=env_int('MULTIAGENT_KNN_VARIANT_FETCH_MULTIPLIER', env_int('VARIANT_FETCH_MULTIPLIER', 3)),
    knn_max_fetch_size=env_int('MULTIAGENT_KNN_MAX_FETCH_SIZE', 1500),
    knn_score_agg=os.getenv('MULTIAGENT_KNN_SCORE_AGG', 'max'),
    knn_rrf_k=env_int('MULTIAGENT_KNN_RRF_K', 60),
)

print('DEBUG_MODEL   =', DEBUG_MODEL)
print('AGENT2_MODEL  =', os.environ.get('MULTIAGENT_AGENT2_LLM_MODEL', DEBUG_MODEL), 'out=', os.environ.get('MULTIAGENT_AGENT2_MAX_OUTPUT_TOKENS'))
print('AGENT3_MODEL  =', os.environ.get('MULTIAGENT_AGENT3_LLM_MODEL', DEBUG_MODEL), 'out=', os.environ.get('MULTIAGENT_AGENT3_MAX_OUTPUT_TOKENS'))
print('DEBUG_TOPIC   =', debug_topic['topic_id'])
print('INPUT_WORDS   =', len(debug_topic['input_text'].split()))
print('TEXT_COLUMNS  =', TEXT_COLUMNS)

In [ ]:
# Step 1: Agent 1 - query understanding.
import inspect, time

print(f"[Agent1] start topic={debug_topic['topic_id']} model={debug_config.llm_model}", flush=True)
started = time.time()
agent1_kwargs = {
    'input_metadata': debug_topic['input_metadata'],
    'config': debug_config,
    'variant': RUN_VARIANTS[0],
}
if 'verbose' in inspect.signature(run_query_understanding_stage).parameters:
    agent1_kwargs['verbose'] = True
else:
    print('[Agent1] verbose unsupported in this imported module; continuing with start/done logs only.', flush=True)

debug_agent1_state = run_query_understanding_stage(debug_topic['input_text'], **agent1_kwargs)
debug_state = debug_agent1_state

print(f"[Agent1] done seconds={time.time() - started:.1f}", flush=True)
print('agent1_done:', True)
agent1_audit_tail = debug_state.get('audit_log', [])[-3:]
agent1_used_fallback = any(item.get('used_fallback') for item in agent1_audit_tail if isinstance(item, dict))
print('agent1_used_fallback:', agent1_used_fallback)
print('technical_problem:', debug_state.get('technical_problem'))
print('claim_elements:', len(debug_state.get('claim_elements', [])))
print('search_queries:', len(debug_state.get('search_queries', [])))
display(pd.DataFrame(debug_state.get('search_queries', [])))
display(pd.DataFrame(agent1_audit_tail))

In [ ]:
# Step 2: Fixed KNN retrieval + deterministic candidate screening.
debug_retrieval_state = run_retrieval_stage(
    debug_agent1_state,
    config=debug_config,
    include_screening=True,
    include_report=False,
)
debug_state = debug_retrieval_state

ctx = debug_state.get('retrieval_context') or {}
print('retrieval_backend:', ctx.get('backend'))
print('retrieval_error  :', ctx.get('error', ''))
print('raw_candidates  :', len(debug_state.get('candidates', [])))
print('screened        :', len(debug_state.get('screened_candidates', [])))
print('agent2_input    : top', os.environ['MULTIAGENT_AGENT2_TOP_DOCS'], 'screened docs')

candidate_rows = []
for cand in (debug_state.get('screened_candidates') or [])[:int(os.environ['MULTIAGENT_AGENT2_TOP_DOCS'])]:
    candidate_rows.append({
        'rank': cand.get('screen_rank') or cand.get('rank'),
        'doc_id': cand.get('doc_id'),
        'score': cand.get('score'),
        'title': cand.get('title'),
        'publication_date': cand.get('publication_date'),
    })
display(pd.DataFrame(candidate_rows))

In [ ]:
# Step 3: Agent 2 - evidence table for the top 3 docs.
debug_agent2_state = run_evidence_stage(
    debug_retrieval_state,
    config=debug_config,
    include_report=False,
)
debug_state = debug_agent2_state

agent2_audit_tail = debug_state.get('audit_log', [])[-5:]
agent2_used_fallback = any(item.get('node') == 'evidence_extraction_agent' and item.get('used_fallback') for item in agent2_audit_tail if isinstance(item, dict))
print('agent2_used_fallback:', agent2_used_fallback)
print('evidence_docs:', len(debug_state.get('evidence', [])))
print('agent3_input : top docs + this evidence table')
evidence_rows = []
for item in debug_state.get('evidence', []):
    for match in item.get('matched_elements', [])[:3]:
        evidence_rows.append({
            'patent_id': item.get('patent_id'),
            'claim_element_vi': match.get('claim_element_vi') or match.get('claim_element'),
            'match_type_vi': match.get('match_type_vi') or match.get('match_type'),
            'section_vi': match.get('section_vi') or match.get('section'),
            'evidence_text': match.get('evidence_text'),
            'gap_or_limitation': match.get('gap_or_limitation'),
        })
display(pd.DataFrame(evidence_rows[:20]))
display(pd.DataFrame(agent2_audit_tail))

In [ ]:
# Step 4: Agent 3 - prior-art analysis from top 3 docs + evidence table.
debug_agent3_state = run_analysis_stage(
    debug_agent2_state,
    config=debug_config,
    include_coverage=True,
)
debug_state = debug_agent3_state

agent3_audit_tail = debug_state.get('audit_log', [])[-6:]
agent3_used_fallback = any(item.get('node') == 'prior_art_analysis_agent' and item.get('used_fallback') for item in agent3_audit_tail if isinstance(item, dict))
print('agent3_used_fallback:', agent3_used_fallback)
analysis = debug_state.get('analysis') or {}
coverage = debug_state.get('coverage') or {}
acceptance = analysis.get('acceptance_assessment') or {}
print('coverage:', json.dumps(coverage, ensure_ascii=False, indent=2))
print('acceptance:', json.dumps(acceptance, ensure_ascii=False, indent=2))
ranked_df = pd.DataFrame(analysis.get('ranked_prior_art', []))
preferred_cols = ['rank', 'patent_id', 'title', 'novelty_risk_vi', 'matched_elements_vi', 'missing_elements_vi', 'claim_overlap_summary', 'limitations']
if not ranked_df.empty:
    ranked_df = ranked_df[[col for col in preferred_cols if col in ranked_df.columns]]
display(ranked_df)
display(pd.DataFrame(agent3_audit_tail))

In [ ]:
# Step 5: Output - finalize state, save JSON, render report.
debug_output_state = run_output_stage(
    debug_agent3_state,
    config=debug_config,
)
debug_state = debug_output_state

output_path = OUTPUT_DIR / f"debug_{slug(debug_topic['topic_id'])}_{slug(DEBUG_MODEL)}.json"
output_path.write_text(json.dumps(jsonable(debug_state), ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved:', output_path)
print('proxy_metrics:', json.dumps(debug_state.get('proxy_metrics', {}), ensure_ascii=False, indent=2))

report = debug_state.get('final_report') or (debug_state.get('analysis') or {}).get('final_report_markdown', '')
if report:
    display(Markdown(report))
else:
    print('No final report was produced.')